# Day 4 · 09 · Data-Quality Checks

Every analysis rests on the assumption that the data is what it claims to be. Data-quality checks test that assumption **explicitly**, before the numbers reach a report.

**In this notebook**
- Loading without silent conversion, and profiling
- The dimensions of data quality
- Finding problems: duplicates, formats, dates, numbers, keys, patterns
- Standardising what can be repaired safely
- A reusable set of checks and a quality report
- Quarantine: separating clean rows from rejected rows, with reasons
- Assertions as a quality gate
- Reconciling against a trusted source

**Dataset:** `orders_raw.csv`, a deliberately messy export with one row per order. `customers.csv` and `order_items.csv` act as trusted reference data.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../datasets").resolve()

# dtype="str" and keep_default_na=False: read exactly what is in the file, convert nothing.
raw = pd.read_csv(DATA / "orders_raw.csv", dtype="str", keep_default_na=False)
customers = pd.read_csv(DATA / "customers.csv")
items = pd.read_csv(DATA / "order_items.csv")

print(raw.shape)
raw.head(6)

(401, 8)


,order_id,customer_id,email,order_date,status,channel,quantity,amount
0,10332,C0201,owen.kelly201@example.com,2024-05-01,completed,mobile,4,374.11
1,10747,C0191,owen.brown191@example.com,2024-09-09,completed,web,5,121.50
2,10592,C0041,vikram.bernard41@example.com,2024-07-18,completed,web,6,270.85
3,11484,C9999,grace.roy181@example.com,2025-03-02,SHIPPED,mobile,2,14375.00
4,10257,C0036,kavya.weber36@example.com,2024-04-06,completed,web,5,
5,10355,C0201,owen.kelly201@example.com,2024-05-09,shipped,web,5,$144.00


## 1. Profile first
Before writing any rule, look at each column: how many blanks, how many distinct values, what the values look like.

In [2]:
pd.DataFrame(
    {
        "blank": (raw == "").sum(),
        "distinct": raw.nunique(),
        "shortest": raw.apply(lambda column: column.str.len().min()),
        "longest": raw.apply(lambda column: column.str.len().max()),
        "example": raw.iloc[5],
    }
)

,blank,distinct,shortest,longest,example
order_id,0,380,5,5,10355
customer_id,5,143,0,5,C0201
email,20,135,0,31,owen.kelly201@example.com
order_date,8,308,0,12,2024-05-09
status,0,13,7,11,shipped
channel,0,3,3,6,web
quantity,10,18,0,3,5
amount,23,341,0,8,$144.00


## 2. The dimensions of data quality
| Dimension | Question | Example in this file |
| --- | --- | --- |
| Completeness | Is the value there? | blank customer id, date, amount |
| Uniqueness | Is each record present once? | repeated order ids |
| Validity | Does it have an allowed format or value? | unknown status, malformed email |
| Consistency | Is the same thing written the same way? | `Completed`, ` completed`, `COMPLETED` |
| Accuracy / range | Is the value plausible? | negative amounts, zero quantity |
| Referential integrity | Does the key exist in the reference table? | customer `C9999` |
| Timeliness | Is the date possible and current? | orders dated in 2030 |

The next sections look at each kind of problem in the raw text.

## 3. Uniqueness
Two different situations hide behind "duplicate":
- **Exact duplicates**: the whole row is repeated. Dropping the copies is safe.
- **Conflicting duplicates**: the same key with different values. Nothing in the data says which row is right, so these need a decision.

In [3]:
exact_copy = raw.duplicated(keep="first")
print("exact duplicate rows:", exact_copy.sum())

deduplicated = raw[~exact_copy]
per_id = deduplicated["order_id"].value_counts()
conflicting_ids = per_id[per_id > 1].index
print("order ids that still appear more than once:", len(conflicting_ids))

deduplicated[deduplicated["order_id"].isin(conflicting_ids)].sort_values("order_id").head(4)

exact duplicate rows: 15
order ids that still appear more than once: 6


,order_id,customer_id,email,order_date,status,channel,quantity,amount
2,10592,C0041,vikram.bernard41@example.com,2024-07-18,completed,web,6,270.85
373,10592,C0041,vikram.bernard41@example.com,2024-07-18,completed,web,6,1.00
6,10603,C0201,owen.kelly201@example.com,2024-07-23,Cancelled,mobile,3,1.00
17,10603,C0201,owen.kelly201@example.com,2024-07-23,Cancelled,mobile,3,28.80


## 4. Consistency and validity of categories
`value_counts` on a text column exposes spelling variants at a glance. Trimming spaces, lower-casing and mapping known synonyms collapses them to the allowed set.

In [4]:
ALLOWED_STATUS = {"completed", "shipped", "cancelled", "returned"}

status = raw["status"].str.strip().str.lower().replace({"canceled": "cancelled"})

print("distinct values as loaded:", raw["status"].nunique(), "->", sorted(raw["status"].unique())[:5], "...")
print("after standardising      :", status.nunique(), "->", status.value_counts().to_dict())
print("outside the allowed set  :", (~status.isin(ALLOWED_STATUS)).sum())

distinct values as loaded: 13 -> [' RETURNED', ' completed ', 'COMPLETED', 'Cancelled', 'Completed'] ...
after standardising      : 4 -> {'completed': 319, 'cancelled': 35, 'shipped': 30, 'returned': 17}
outside the allowed set  : 0


## 5. Dates in several formats
One column, three date formats. Parse each format **explicitly** and combine the results. Guessing formats row by row is slow and can silently swap day and month.

In [5]:
DATE_FORMATS = {"2024-03-09": "%Y-%m-%d", "09/03/2024": "%d/%m/%Y", "Mar 09, 2024": "%b %d, %Y"}


def parse_dates(text):
    parsed = [pd.to_datetime(text, format=fmt, errors="coerce") for fmt in DATE_FORMATS.values()]
    result = parsed[0]
    for other in parsed[1:]:
        result = result.fillna(other)
    return result


for example, fmt in DATE_FORMATS.items():
    matched = pd.to_datetime(raw["order_date"], format=fmt, errors="coerce").notna().sum()
    print(f"{fmt:<10} like {example:<13}: {matched} rows")

order_date = parse_dates(raw["order_date"])
CUTOFF = pd.Timestamp("2025-06-30")   # the export was taken on this date

print("blank                      :", (raw["order_date"] == "").sum())
print("not blank but unparseable  :", (order_date.isna() & (raw["order_date"] != "")).sum())
print("after the export date      :", (order_date > CUTOFF).sum(), "->", sorted(order_date[order_date > CUTOFF].dt.year.unique().tolist()))

%Y-%m-%d   like 2024-03-09   : 285 rows
%d/%m/%Y   like 09/03/2024   : 59 rows
%b %d, %Y  like Mar 09, 2024 : 49 rows
blank                      : 8
not blank but unparseable  : 0
after the export date      : 8 -> [2030, 2031]


## 6. Numbers stored as text
`amount` contains currency symbols and thousands separators, so a direct conversion fails for those rows. Remove the formatting, convert with `errors="coerce"`, then count what is still missing.

In [6]:
naive_amount = pd.to_numeric(raw["amount"], errors="coerce")
amount = pd.to_numeric(raw["amount"].str.replace(r"[$,]", "", regex=True), errors="coerce")
quantity = pd.to_numeric(raw["quantity"], errors="coerce")

print("amount  : blank", (raw["amount"] == "").sum(), "| unreadable without cleaning", naive_amount.isna().sum(), "| after cleaning", amount.isna().sum())
print("amount  : negative", (amount < 0).sum())
print("quantity: blank or text", quantity.isna().sum(), "| zero or negative", (quantity <= 0).sum())

amount  : blank 23 | unreadable without cleaning 148 | after cleaning 23
amount  : negative 13
quantity: blank or text 19 | zero or negative 15


### Outliers: a statistical screen and a business rule
A common screen flags values far outside the interquartile range (IQR). It needs no domain knowledge, but it cannot tell an error from a genuinely large order.

A **business rule** is sharper. No product costs more than the most expensive item in the catalogue, so an order's amount can never exceed `quantity × highest unit price`.

`order_items.csv` holds the true order totals, which lets us see how each method performs.

In [7]:
true_total = (items["quantity"] * items["unit_price"] * (1 - items["discount"])).groupby(items["order_id"]).sum()
expected_amount = raw["order_id"].astype("int64").map(true_total)
is_wrong = amount.notna() & ~np.isclose(amount, expected_amount, atol=0.01)

q1, q3 = amount.quantile([0.25, 0.75])
iqr_limit = q3 + 3 * (q3 - q1)
flag_iqr = amount > iqr_limit

max_price = pd.read_csv(DATA / "products.csv")["unit_price"].max()
flag_rule = (quantity > 0) & (amount > quantity * max_price)

print(f"IQR limit: {iqr_limit:,.2f} | highest unit price: {max_price}")
pd.DataFrame(
    {
        "flagged": {"IQR screen": flag_iqr.sum(), "business rule": flag_rule.sum()},
        "flagged_and_really_wrong": {"IQR screen": (flag_iqr & is_wrong).sum(), "business rule": (flag_rule & is_wrong).sum()},
        "flagged_but_correct": {"IQR screen": (flag_iqr & ~is_wrong).sum(), "business rule": (flag_rule & ~is_wrong).sum()},
    }
)

IQR limit: 856.70 | highest unit price: 199.0


,flagged,flagged_and_really_wrong,flagged_but_correct
IQR screen,21,17,4
business rule,16,16,0


The IQR screen also flags genuine large orders (`flagged_but_correct`). The business rule flags only real errors. It cannot judge rows whose quantity is itself invalid; those rows fail the quantity checks instead.

## 7. Referential integrity
Every `customer_id` should exist in the customer table. Standardise the key first, otherwise formatting differences are counted as missing customers.

In [8]:
known_ids = set(customers["customer_id"])
customer_id = raw["customer_id"].str.strip().str.upper()

print("blank                         :", (raw["customer_id"] == "").sum())
print("not found, as loaded          :", ((raw["customer_id"] != "") & ~raw["customer_id"].isin(known_ids)).sum())
print("not found, after standardising:", ((customer_id != "") & ~customer_id.isin(known_ids)).sum())
print("ids not found                 :", sorted(customer_id[(customer_id != "") & ~customer_id.isin(known_ids)].unique()))

blank                         : 5
not found, as loaded          : 37
not found, after standardising: 12
ids not found                 : ['C9999']


## 8. Patterns
A regular expression checks the **shape** of a value. `str.fullmatch` requires the whole value to match.

This pattern asks for `something@something.something` with no spaces. It checks the format only, not whether the address exists.

In [9]:
EMAIL_PATTERN = r"[^@\s]+@[^@\s]+\.[^@\s]+"

email = raw["email"].str.strip().str.lower()
well_formed = email.str.fullmatch(EMAIL_PATTERN, na=False)

print("blank    :", (email == "").sum())
print("malformed:", ((email != "") & ~well_formed).sum())
email[(email != "") & ~well_formed].head(3).tolist()

blank    : 20
malformed: 14


['ryan.johnson109 at example.com', 'noah.dubois166 at example.com', 'grace.roy181 at example.com']

## 9. Standardise, then check
Now the steps become a pipeline.

**Step 1: standardise.** Apply only repairs that cannot change the meaning: trim spaces, fix case, map known synonyms, remove currency formatting, parse known date formats, drop exact duplicates. Anything that cannot be converted becomes missing.

The `repair=False` option skips those repairs. It shows what a naive type conversion would give, for comparison.

In [10]:
def standardise(frame, repair=True):
    """Return a typed copy of the raw export."""
    customer = frame["customer_id"]
    status = frame["status"]
    amount = frame["amount"]
    email = frame["email"]
    if repair:
        customer = customer.str.strip().str.upper()
        status = status.str.strip().str.lower().replace({"canceled": "cancelled"})
        amount = amount.str.replace(r"[$,]", "", regex=True)
        email = email.str.strip().str.lower()
        order_date = parse_dates(frame["order_date"])
    else:
        order_date = pd.to_datetime(frame["order_date"], format="%Y-%m-%d", errors="coerce")

    return pd.DataFrame(
        {
            "order_id": pd.to_numeric(frame["order_id"], errors="coerce").astype("Int64"),
            "customer_id": customer.replace("", pd.NA),
            "email": email.replace("", pd.NA),
            "order_date": order_date,
            "status": status,
            "channel": frame["channel"].str.strip().str.lower(),
            "quantity": pd.to_numeric(frame["quantity"], errors="coerce").astype("Int64"),
            "amount": pd.to_numeric(amount, errors="coerce"),
        },
        index=frame.index,
    )


typed = standardise(raw.drop_duplicates())
print(len(raw), "raw rows ->", len(typed), "after dropping exact duplicates")
typed.dtypes

401 raw rows -> 386 after dropping exact duplicates


order_id                Int64
customer_id               str
email                     str
order_date     datetime64[us]
status                    str
channel                   str
quantity                Int64
amount                float64
dtype: object

**Step 2: checks.** Each check is a name, a dimension, a severity and a boolean mask that is `True` where a row **fails**.

- `error`: the row cannot be used.
- `warning`: the row is usable, but the problem is worth reporting.

Keeping every check in one function makes the rule set easy to read, review and extend.

In [11]:
def build_checks(frame):
    """Return {check name: (dimension, severity, mask of failing rows)}."""
    quantity_ok = frame["quantity"].gt(0).fillna(False).astype("bool")
    known = frame["customer_id"].isin(known_ids)
    email_ok = frame["email"].str.fullmatch(EMAIL_PATTERN, na=False).astype("bool")
    expected_email = frame["customer_id"].map(customers.set_index("customer_id")["email"])

    checks = {
        "customer_id present": ("completeness", "error", frame["customer_id"].isna()),
        "order_date readable": ("completeness", "error", frame["order_date"].isna()),
        "quantity readable": ("completeness", "error", frame["quantity"].isna()),
        "amount readable": ("completeness", "error", frame["amount"].isna()),
        "order_id unique": ("uniqueness", "error", frame["order_id"].duplicated(keep=False)),
        "status allowed": ("validity", "error", ~frame["status"].isin(ALLOWED_STATUS)),
        "quantity above zero": ("range", "error", frame["quantity"].notna() & ~quantity_ok),
        "amount not negative": ("range", "error", frame["amount"] < 0),
        "amount plausible for quantity": ("range", "error", quantity_ok & (frame["amount"] > frame["quantity"].astype("float64") * max_price)),
        "customer exists": ("referential integrity", "error", frame["customer_id"].notna() & ~known),
        "order_date not in the future": ("timeliness", "error", frame["order_date"] > CUTOFF),
        "email present": ("completeness", "warning", frame["email"].isna()),
        "email well formed": ("validity", "warning", frame["email"].notna() & ~email_ok),
        "email matches customer record": ("consistency", "warning", email_ok & expected_email.notna() & (frame["email"] != expected_email)),
    }
    return {name: (dimension, severity, mask.fillna(False).astype("bool")) for name, (dimension, severity, mask) in checks.items()}


def quality_report(frame):
    rows = [
        {"check": name, "dimension": dimension, "severity": severity, "failed": int(mask.sum()), "failed_pct": round(100 * mask.mean(), 1)}
        for name, (dimension, severity, mask) in build_checks(frame).items()
    ]
    return pd.DataFrame(rows).set_index("check")


report = quality_report(typed)
report

,dimension,severity,failed,failed_pct
check,,,,
customer_id present,completeness,error,5,1.3
order_date readable,completeness,error,7,1.8
quantity readable,completeness,error,18,4.7
amount readable,completeness,error,23,6.0
order_id unique,uniqueness,error,12,3.1
status allowed,validity,error,0,0.0
quantity above zero,range,error,15,3.9
amount not negative,range,error,12,3.1
amount plausible for quantity,range,error,16,4.1


### How much was only formatting?
Run the same checks on a naive conversion of the file. The difference between the two columns is what standardising and dropping the 15 exact duplicates repaired. What remains in `after` are real data problems.

In [12]:
before = quality_report(standardise(raw, repair=False))

pd.DataFrame(
    {
        "severity": report["severity"],
        "before_standardising": before["failed"],
        "after_standardising": report["failed"],
    }
)

,severity,before_standardising,after_standardising
check,,,
customer_id present,error,5,5
order_date readable,error,116,7
quantity readable,error,19,18
amount readable,error,148,23
order_id unique,error,42,12
status allowed,error,181,0
quantity above zero,error,15,15
amount not negative,error,13,12
amount plausible for quantity,error,16,16


## 10. Quarantine instead of delete
Rows that fail an `error` check are moved to a **quarantine** table together with the reasons. Nothing is thrown away:
- the clean table feeds the analysis
- the quarantine table goes back to whoever owns the source
- the counts of both are reported

In [13]:
checks = build_checks(typed)
flags = pd.DataFrame({name: mask for name, (_, _, mask) in checks.items()})
error_checks = [name for name, (_, severity, _) in checks.items() if severity == "error"]
warning_checks = [name for name, (_, severity, _) in checks.items() if severity == "warning"]

rejected = flags[error_checks].any(axis=1)

clean = typed[~rejected].copy()
clean["warnings"] = flags.loc[~rejected, warning_checks].sum(axis=1)

quarantine = typed[rejected].copy()
quarantine["reasons"] = flags.loc[rejected, error_checks].apply(lambda row: "; ".join(row.index[row.to_numpy()]), axis=1)

print(f"clean: {len(clean)} rows | quarantined: {len(quarantine)} rows | clean rows with a warning: {(clean['warnings'] > 0).sum()}")
quarantine["reasons"].value_counts().head(6)

clean: 269 rows | quarantined: 117 rows | clean rows with a warning: 20


reasons
amount readable                  19
quantity readable                14
quantity above zero              14
amount plausible for quantity    14
order_id unique                  12
amount not negative              10
Name: count, dtype: int64

In [14]:
quarantine[["order_id", "customer_id", "order_date", "quantity", "amount", "reasons"]].head(5)

,order_id,customer_id,order_date,quantity,amount,reasons
2,10592,C0041,2024-07-18,6,270.85,order_id unique
3,11484,C9999,2025-03-02,2,14375.00,amount plausible for quantity; customer exists
4,10257,C0036,2024-04-06,5,NaN,amount readable
6,10603,C0201,2024-07-23,3,1.00,order_id unique
9,10125,C9999,2024-02-19,1,21.50,customer exists


## 11. Assertions: a quality gate
State what must be true of the clean table and stop the pipeline if it is not. An `assert` that fails loudly today is cheaper than a wrong figure discovered next month.

In [15]:
assert clean["order_id"].is_unique, "duplicate order ids"
assert clean[["customer_id", "order_date", "quantity", "amount"]].notna().all().all(), "missing required values"
assert clean["customer_id"].isin(known_ids).all(), "unknown customers"
assert clean["status"].isin(ALLOWED_STATUS).all(), "unexpected status"
assert (clean["quantity"] > 0).all() and (clean["amount"] >= 0).all(), "values out of range"
assert clean["order_date"].between("2024-01-01", CUTOFF).all(), "dates out of range"

print("all assertions passed for", len(clean), "clean rows")

all assertions passed for 269 clean rows


## 12. Reconcile against a trusted source
Rules can only test plausibility. The strongest check compares the data with an independent, trusted source. Here `order_items.csv` gives the true quantity and amount of every order.

In [16]:
truth = items.assign(amount=items["quantity"] * items["unit_price"] * (1 - items["discount"])).groupby("order_id").agg(
    true_quantity=("quantity", "sum"), true_amount=("amount", "sum")
)


def reconcile(frame):
    joined = frame.merge(truth, left_on="order_id", right_index=True, how="left")
    amount_ok = np.isclose(joined["amount"].astype("float64"), joined["true_amount"], atol=0.01)
    quantity_ok = (joined["quantity"] == joined["true_quantity"]).fillna(False).astype("bool")
    return pd.Series({"rows": len(joined), "amount_matches": int(amount_ok.sum()), "quantity_matches": int(quantity_ok.sum())})


pd.DataFrame({"clean": reconcile(clean), "quarantine": reconcile(quarantine)})

,clean,quarantine
rows,269,117
amount_matches,269,59
quantity_matches,269,84


Every clean row agrees with the source on both fields. In quarantine, about half of the rows have a correct amount and were rejected for a different reason, which is why the reasons are kept with each row.

In [17]:
# What the cleaning changes in a headline number.
naive_total = pd.to_numeric(raw["amount"], errors="coerce").sum()
clean_total = clean["amount"].sum()

print(f"total amount, naive conversion of the raw file: {naive_total:>14,.2f}")
print(f"total amount, clean rows                      : {clean_total:>14,.2f}")

total amount, naive conversion of the raw file:     404,491.69
total amount, clean rows                      :      53,611.10


## 13. The same idea in Day 2's `datakit`
The `datakit` package from Day 2 applies this pattern row by row in plain Python: parse, validate, and collect rejects with a reason. Its default rule set is smaller than the one above (no reference, date-range or plausibility checks) and it keeps the first row of a duplicated order id.

Applying that smaller rule set with pandas:

In [18]:
everything = standardise(raw)   # all rows, exact duplicates included

passes = (
    everything[["customer_id", "order_date", "quantity", "amount"]].notna().all(axis=1)
    & everything["status"].isin(ALLOWED_STATUS)
    & everything["quantity"].gt(0).fillna(False).astype("bool")
    & (everything["amount"] >= 0)
)
accepted = everything[passes].drop_duplicates("order_id")

print("rows accepted with the datakit rule set:", len(accepted), "| rejected:", len(everything) - len(accepted))

rows accepted with the datakit rule set: 306 | rejected: 95


These are the same counts that `datakit` reports for this file in the Day 2 project notebook: two implementations of one rule set, checking each other.


## Key takeaways
- Load raw data as text and profile it before trusting any conversion.
- Standardise formats first; then the checks measure real problems.
- Express each rule as a named boolean mask. A dictionary of masks is a test suite for your data.
- Separate `error` from `warning`, and quarantine with reasons instead of deleting.
- Guard the output with assertions, and reconcile with a trusted source when one exists.
- Report the counts: rows in, rows clean, rows rejected and why.

## Practice
1. Add a check that `channel` is one of `web`, `mobile`, `store`.
2. Add a warning for orders placed before the customer's `signup_date`.
3. Change the duplicate rule to keep the first row of a conflicting order id. How do the clean count and the reconciliation change?
4. Wrap sections 9 to 11 in a single function `clean_orders(raw) -> (clean, quarantine, report)`.